In [1]:
import time
from collections import Counter
from datetime import UTC, datetime, timedelta
from pathlib import Path

import boto3
from dotenv import dotenv_values, unset_key

import sys
sys.path.insert(0, "..")  # the helper is in labs/week3
import week3day4_aws as aws

In [2]:
session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

Working as arn:aws:iam::123456789012:user/undercover in us-east-1


In [3]:
lambda_client = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
kms = session.client("kms")
s3 = session.client("s3")
secrets_manager = session.client("secretsmanager")
agentcore = session.client("bedrock-agentcore-control")
explorer = session.client("resource-explorer-2")
budgets = session.client("budgets")

In [4]:
LOG_GROUPS = ["/aws/lambda/undercover", "/aws/bedrock-agentcore/runtimes/undercover"]


def token_vault_key():
    """The key of yours that the token vault uses, if it isn't on AWS's own key."""
    key = agentcore.get_token_vault(tokenVaultId="default").get("kmsConfiguration", {})
    return [key["kmsKeyArn"]] if key.get("keyType") == "CustomerManagedKey" else []


def find_undercover():
    """Everything of UnderCover's on the account, found by name, service by service."""
    functions = [f["FunctionName"] for page in lambda_client.get_paginator("list_functions").paginate() for f in page["Functions"]]
    providers = [p["name"] for page in agentcore.get_paginator("list_api_key_credential_providers").paginate() for p in page["credentialProviders"]]
    memories = [m for page in agentcore.get_paginator("list_memories").paginate() for m in page["memories"]]
    roles = [r["RoleName"] for page in iam.get_paginator("list_roles").paginate() for r in page["Roles"]]
    log_pages = [logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=prefix) for prefix in LOG_GROUPS]
    secrets = [s["Name"] for page in secrets_manager.get_paginator("list_secrets").paginate() for s in page["SecretList"]]
    aliases = [a["AliasName"] for page in kms.get_paginator("list_aliases").paginate() for a in page["Aliases"]]
    return {
        "Lambda functions": [name for name in functions if name.startswith("undercover")],
        "Runtimes": [r["agentRuntimeName"] for r in aws.find_runtimes("undercover")],
        "Gateways": [g["name"] for g in aws.find_gateways()],
        "Policy engines": [e["name"] for e in aws.find_policy_engines()],
        "Credential providers": [name for name in providers if name.startswith("undercover")],
        "Memories": [m["id"] for m in memories if "undercover" in m["id"] and m["status"] != "DELETING"],
        "Stacks": aws.find_stacks(),
        "IAM roles": [name for name in roles if name.startswith(("undercover", "AgentCore-undercover", "cdk-hnb659fds"))],
        "Log groups": [g["logGroupName"] for pages in log_pages for page in pages for g in page["logGroups"]],
        "Secrets": [name for name in secrets if "undercover" in name],
        "S3 buckets": [b["Name"] for b in s3.list_buckets()["Buckets"] if b["Name"].startswith("cdk-hnb659fds")],
        "Encryption keys": [name for name in aliases if name.startswith("alias/cdk-hnb659fds")],
        "Token vault key of yours": token_vault_key(),
    }


def show_undercover():
    """Print what's left of UnderCover on the account, by service."""
    for label, names in find_undercover().items():
        print(f"{label + ':':<26}{', '.join(names) or 'none'}")


show_undercover()

Lambda functions:         undercover-context, undercover-app
Runtimes:                 undercover_default_copilot, undercover_default_agent
Gateways:                 undercover-gateway
Policy engines:           undercover_policy_engine
Credential providers:     undercover-app-key, undercover_default_DATABASE_URL, undercover_default_OPENROUTER_API_KEY, undercover_default_UNDERCOVER_API_KEY
Memories:                 undercover_copilot_memory-EYD73DChMs, undercover_memory-vYyePd72Ze
Stacks:                   AgentCore-undercover-default, CDKToolkit
IAM roles:                AgentCore-undercover-defa-ApplicationRuntimeAgentRu-by5ACnEIAVZy, AgentCore-undercover-defa-ApplicationRuntimeCopilot-KDSF9rW4OoKe, cdk-hnb659fds-cfn-exec-role-123456789012-us-east-1, cdk-hnb659fds-deploy-role-123456789012-us-east-1, cdk-hnb659fds-file-publishing-role-123456789012-us-east-1, cdk-hnb659fds-image-publishing-role-123456789012-us-east-1, cdk-hnb659fds-lookup-role-123456789012-us-east-1, undercover-app-role

In [5]:
def show_everything():
    """Count every resource Resource Explorer knows of, by Region and type, and list
    any with undercover in its name."""
    try:
        pages = explorer.get_paginator("list_resources").paginate()
        resources = [r for page in pages for r in page["Resources"]]
    except explorer.exceptions.AccessDeniedException:
        print("Your IAM user isn't allowed to use Resource Explorer. The course policy from Day 1, labs/undercover-course-policy.json, allows it.")
        return
    except explorer.exceptions.UnauthorizedException:
        print("Resource Explorer isn't set up in this Region yet. Open it in the AWS console, signed in as root, then run this cell again.")
        return
    counts = Counter((r["Region"], r["ResourceType"]) for r in resources)
    for (resource_region, resource_type), count in sorted(counts.items()):
        print(f"{resource_region:<16}{resource_type:<50}{count}")
    print(f"\n{len(resources)} resources in all")
    ours = sorted(r["Arn"] for r in resources if "undercover" in r["Arn"].lower())
    print(f"\nWith undercover in the name: {len(ours)}")
    for arn in ours:
        print(f"  {arn}")


show_everything()

global          budgets:budget                                    3
global          ce:anomalymonitor                                 1
global          ce:anomalysubscription                            1
global          iam:group                                         1
global          iam:mfa                                           1
global          iam:policy                                        1
global          iam:role                                          20
global          iam:user                                          1
us-east-1       apigateway:restapis                               1
us-east-1       apigateway:restapis/deployments                   5
us-east-1       apigateway:restapis/resources                     2
us-east-1       apigateway:restapis/resources/methods             2
us-east-1       apigateway:restapis/stages                        1
us-east-1       apprunner:autoscalingconfiguration                1
us-east-1       athena:datacatalog             

In [6]:
LOCAL_ENV = Path("../../../agentcore/.env.local")
SETTINGS = ["UNDERCOVER_API", "UNDERCOVER_GATEWAY_URL", "UNDERCOVER_MEMORY_ID", "UNDERCOVER_COPILOT_MEMORY_ID"]

present = dotenv_values(LOCAL_ENV) if LOCAL_ENV.exists() else {}
for name in SETTINGS:
    if name in present:
        unset_key(LOCAL_ENV, name)
        print(f"Took {name} out of {LOCAL_ENV}")

Took UNDERCOVER_API out of ..\agentcore\.env.local
Took UNDERCOVER_GATEWAY_URL out of ..\agentcore\.env.local
Took UNDERCOVER_MEMORY_ID out of ..\agentcore\.env.local
Took UNDERCOVER_COPILOT_MEMORY_ID out of ..\agentcore\.env.local


In [7]:
runtime_log_groups = [
    group["logGroupName"]
    for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix="/aws/bedrock-agentcore/runtimes/undercover")
    for group in page["logGroups"]
]

for name in runtime_log_groups:
    logs.delete_log_group(logGroupName=name)
    print(f"Deleted log group {name}")

Deleted log group /aws/bedrock-agentcore/runtimes/undercover_default_agent-kv5KDa3Due-DEFAULT
Deleted log group /aws/bedrock-agentcore/runtimes/undercover_default_copilot-F4DRozEWFg-DEFAULT


In [8]:
aws.tear_down()

Deleted target undercover-app
Deleted target undercover-context
Deleted gateway undercover-gateway
Deleted policy undercover_allow_app_tools
Deleted policy undercover_allow_context_tools
Deleted policy undercover_payment_limit
Deleted policy engine undercover_policy_engine
Deleted credential provider undercover-app-key
Deleted memory undercover_memory-vYyePd72Ze
Deleted role undercover-gateway-role


In [9]:
memories = [m for page in agentcore.get_paginator("list_memories").paginate() for m in page["memories"]]

for memory in memories:
    if "undercover" in memory["id"] and memory["status"] != "DELETING":
        agentcore.delete_memory(memoryId=memory["id"])
        print(f"Deleted memory {memory['id']}")

Deleted memory undercover_copilot_memory-EYD73DChMs


In [10]:
functions = [f["FunctionName"] for page in lambda_client.get_paginator("list_functions").paginate() for f in page["Functions"]]

for name in functions:
    if name.startswith("undercover"):
        lambda_client.delete_function(FunctionName=name)
        print(f"Deleted function {name}")

for role_name in aws.find_roles():
    aws.delete_role(role_name)

function_log_groups = [
    group["logGroupName"]
    for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix="/aws/lambda/undercover")
    for group in page["logGroups"]
]

for name in function_log_groups:
    logs.delete_log_group(logGroupName=name)
    print(f"Deleted log group {name}")

Deleted function undercover-context
Deleted function undercover-app
Deleted role undercover-app-role
Deleted role undercover-context-role
Deleted log group /aws/lambda/undercover-app
Deleted log group /aws/lambda/undercover-context


In [11]:
cloudformation = session.client("cloudformation")
stacks = [stack for page in cloudformation.get_paginator("describe_stacks").paginate() for stack in page["Stacks"]]
deployed_through_it = [s["StackName"] for s in stacks if "cdk-hnb659fds-cfn-exec-role" in s.get("RoleARN", "") and s["StackName"] != "CDKToolkit"]

if deployed_through_it:
    print(f"Keeping the bootstrap, which these stacks still need: {', '.join(deployed_through_it)}")
    print("Delete them first, then run this cell again. For an AgentCore project, that's agentcore remove all --yes and agentcore deploy --yes in its folder, as in Part 2.")
else:
    aws.delete_bootstrap()

Keeping the bootstrap, which these stacks still need: AgentCore-undercover-default
Delete them first, then run this cell again. For an AgentCore project, that's agentcore remove all --yes and agentcore deploy --yes in its folder, as in Part 2.


In [12]:
CLI_KEY = "AgentCore Identity encryption key"

vault_key = agentcore.get_token_vault(tokenVaultId="default").get("kmsConfiguration", {})

if vault_key.get("keyType") != "CustomerManagedKey":
    print("The token vault uses AWS's own key, so there's nothing to delete")
elif kms.describe_key(KeyId=vault_key["kmsKeyArn"])["KeyMetadata"].get("Description", "").startswith(CLI_KEY):
    agentcore.set_token_vault_cmk(tokenVaultId="default", kmsConfiguration={"keyType": "ServiceManagedKey"})
    kms.schedule_key_deletion(KeyId=vault_key["kmsKeyArn"], PendingWindowInDays=7)
    print(f"The token vault is back on AWS's key, and {vault_key['kmsKeyArn']} will be deleted in 7 days")
else:
    print(f"The token vault uses {vault_key['kmsKeyArn']}, which the CLI didn't make, so we leave it")

The token vault uses AWS's own key, so there's nothing to delete


In [13]:
time.sleep(10)
show_undercover()

Lambda functions:         none
Runtimes:                 undercover_default_copilot, undercover_default_agent
Gateways:                 none
Policy engines:           none
Credential providers:     undercover_default_DATABASE_URL, undercover_default_OPENROUTER_API_KEY, undercover_default_UNDERCOVER_API_KEY
Memories:                 none
Stacks:                   AgentCore-undercover-default, CDKToolkit
IAM roles:                AgentCore-undercover-defa-ApplicationRuntimeAgentRu-by5ACnEIAVZy, AgentCore-undercover-defa-ApplicationRuntimeCopilot-KDSF9rW4OoKe, cdk-hnb659fds-cfn-exec-role-123456789012-us-east-1, cdk-hnb659fds-deploy-role-123456789012-us-east-1, cdk-hnb659fds-file-publishing-role-123456789012-us-east-1, cdk-hnb659fds-image-publishing-role-123456789012-us-east-1, cdk-hnb659fds-lookup-role-123456789012-us-east-1
Log groups:               /aws/bedrock-agentcore/runtimes/undercover_default_agent-kv5KDa3Due-DEFAULT
Secrets:                  bedrock-agentcore-identity!default/api

In [14]:
show_everything()

global          budgets:budget                                    3
global          ce:anomalymonitor                                 1
global          ce:anomalysubscription                            1
global          iam:group                                         1
global          iam:mfa                                           1
global          iam:policy                                        1
global          iam:role                                          20
global          iam:user                                          1
us-east-1       apigateway:restapis                               1
us-east-1       apigateway:restapis/deployments                   5
us-east-1       apigateway:restapis/resources                     2
us-east-1       apigateway:restapis/resources/methods             2
us-east-1       apigateway:restapis/stages                        1
us-east-1       apprunner:autoscalingconfiguration                1
us-east-1       athena:datacatalog             

In [15]:
ALL_SPEND = "undercover-daily-spend"
AFTER_CREDITS = "undercover-daily-spend-after-credits"


def daily_spend(name):
    """Each day's spend on one of monitor.py's budgets by UTC date, and when AWS last updated it."""
    now = datetime.now(UTC)
    period = {"Start": now - timedelta(days=30), "End": now}
    pages = budgets.get_paginator("describe_budget_performance_history").paginate(AccountId=account, BudgetName=name, TimePeriod=period)
    spend = {}
    for page in pages:
        for day in page["BudgetPerformanceHistory"]["BudgetedAndActualAmountsList"]:
            if "ActualAmount" in day:
                spend[day["TimePeriod"]["Start"].astimezone(UTC).date()] = float(day["ActualAmount"]["Amount"])
    budget = budgets.describe_budget(AccountId=account, BudgetName=name)["Budget"]
    if "CalculatedSpend" in budget:
        spend[now.date()] = float(budget["CalculatedSpend"]["ActualSpend"]["Amount"])
    return spend, budget["LastUpdatedTime"]


today = datetime.now(UTC).date()
first_day = today - timedelta(days=29)

for label, name in [("Including what credits paid for", ALL_SPEND), ("What you pay after credits", AFTER_CREDITS)]:
    try:
        spend, updated = daily_spend(name)
    except budgets.exceptions.NotFoundException:
        print(f"There's no budget {name} yet. Run uv run labs/week3/monitor.py once, which creates it.")
        continue
    month = sum(amount for date, amount in spend.items() if date >= first_day)
    print(f"{label}: today ${spend.get(today, 0.0):,.2f}, last 30 days ${month:,.2f}")
    print(f"  AWS last updated this at {updated.astimezone():%H:%M}")

Including what credits paid for: today $0.00, last 30 days $0.09
  AWS last updated this at 14:07
What you pay after credits: today $0.00, last 30 days $0.09
  AWS last updated this at 14:12
